## Acidentes de Trânsito nas Rodovias Federais Brasileiras
**Nome:** Leonardo Quederoli Leme - **RA:** 241020409<br>
**Nome:** Luís Fernando das Chagas - **RA:** 241020034

## Contexto do problema
A malha viária federal estende-se por dezenas de milhares de quilômetros, abrangendo realidades geográficas, climáticas e de tráfego profundamente heterogêneas. O enfrentamento da violência no trânsito por parte dos órgãos fiscalizadores enfrenta um gargalo estrutural: a assimetria entre a extensão territorial a ser coberta e a limitação dos recursos operacionais disponíveis, que incluem contingente de agentes, viaturas, radares móveis e etilômetros.

A distribuição tradicional da fiscalização por vezes baseou-se em percepções empíricas ou no patrulhamento ostensivo genérico, modelos que demonstram limitação para conter sinistros em janelas específicas de vulnerabilidade. A consolidação do repositório DATATRAN/PRF possibilita uma transição metodológica para a Segurança Viária Baseada em Evidências (Data-Driven Policing). Por meio da exploração sistemática desses dados e de técnicas de segmentação, torna-se viável transformar dados históricos em inteligência tática, fundamentando a tomada de decisão sobre onde e quando intervir.       


## Problema central
O problema central investigado nesta análise é a distribuição não uniforme da severidade dos acidentes ao longo da malha rodoviária. As ocorrências que resultam em lesões graves ou fatalidades não ocorrem ao acaso; elas tendem a se adensar na interseção de fatores contextuais específicos. Elementos como a transição de iluminação natural (crepúsculo e madrugada), a configuração da via (pista simples e traçados em curva), a ocorrência de intempéries (chuva e neblina) e condutas humanas de risco atuam de forma sinérgica, multiplicando a energia dos impactos.

O propósito deste relatório analítico é decompor os sinistros em suas variáveis explicativas, fornecendo ao órgão de trânsito matrizes claras de priorização para otimizar escalas de serviço, posicionamento de equipamentos e operações ostensivas focadas na mitigação de mortes.

## Limpeza e Preparação dos Dados

Foram selecionados dois datasets com informações conjuntas a respeito de dados de sinistros de trânsito, registrados pela Polícia Rodoviária Federal ao longo do ano de 2025. Um dos datasets apresenta o registro de sinistros agrupados por ocorrência, ou seja, possui um registro para cada sinistro com ocorrência nas rodovias federais do Brasil. O segundo dataset apresenta o registro de sinistros agrupados por pessoa, de modo que mais de um registro possa tratar do mesmo sinistro, dependendo da quantidade de veículos e pessoas envolvidas na ocorrência. Para uma análise abrangente dos dados, as informações de ambos os datasets serão utilizados, visto que os dados de cada registro de sinistro agrupado por pessoa está relacionado a um dos sinistros agrupados por ocorrência através de um ID.

In [17]:
# Importação das bibliotecas necessárias
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Configura como os dataframes serão exibidos
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Carrega os dois datasets que serão utilizados
df1 = pd.read_csv('datatran2025.csv', sep=';', encoding='ISO-8859-1') # Sinistros agrupados por ocorrência
df2 = pd.read_csv('acidentes2025.csv', sep=';', encoding='ISO-8859-1') # Sinistros agrupados por pessoa

### Preencher Dados Faltantes e Corrigir Inconsistências

In [18]:
# Somatório do total de rows NaN em cada coluna
df1.isna().sum()[df1.isna().sum() > 0]

classificacao_acidente     1
regional                   2
delegacia                 22
uop                       38
dtype: int64

In [19]:
# Somatório do total de rows NaN em cada coluna
df2.isna().sum()[df2.isna().sum() > 0]

classificacao_acidente        7
tipo_veiculo               6341
tipo_envolvido            17150
estado_fisico             17150
sexo                      17150
regional                      6
delegacia                    74
uop                         130
dtype: int64

Buscou-se corrigir inconsistências encontradas nos datasets preenchendo os campos com dados faltantes, corrigindo os valores de variáveis derivadas e alterando o nome de algumas colunas para torná-los corretos ou mais compreensíveis. Primeiramente, todos os campos com dados faltantes de variáveis categóricas foram preenchidos com "Não Informado", termo já utilizado no dataset para identificar situações onde não há ciência sobre uma determinada informação do registro.

In [20]:
# Preenche as rows com conteúdo NaN das colunas específicadas, com o termo "Não Informado"
df1[["regional", "delegacia", "uop"]] = df1[["regional", "delegacia", "uop"]].fillna("Não Informado")
df2[["regional", "delegacia", "uop", "tipo_veiculo", "tipo_envolvido", "estado_fisico", "sexo"]] = df2[["regional", "delegacia", "uop", "tipo_veiculo", "tipo_envolvido", "estado_fisico", "sexo"]].fillna("Não Informado")

Para preencher os campos faltantes da coluna "classificacao_acidente" redefiniu-se o valor de todos os elementos da coluna, garantindo que todos obedecessem a mesma lógica e estivessem classificados corretamente. Isso foi feito levando em consideração a quantidade de indivíduos envolvidos e seus estados físicos após o acidente. Sendo o número de mortos acima de 0, o acidente é classificado como "Com Vítimas Fatais". Se não houver nenhuma fatalidade mas houver pelo menos um indivíduo ferido, o acidente passa a ser classificado como "Com Vítimas Feridas". Por fim, caso o acidente não se encaixe em nenhuma das classificações anteriores ele é considerado "Sem Vítimas". Após preencher a coluna no primeiro dataset, seus valores são copiados para as colunas do segundo dataset que correspodem ao mesmo ID da ocorrência.

In [21]:
# Determina um conjunto de condições
conditions = [
    (df1["mortos"] > 0),
    (df1["feridos"] > 0),
]

# Determina um conjunto de escolhas
choices = ["Com Vítimas Fatais", "Com Vítimas Feridas"]

# Preenche a coluna com as escolhas definidas de acordo com cada condição
# Caso nenhuma condição seja cumprida, a coluna é preenchida com o default
df1["classificacao_acidente"] = np.select(conditions, choices, default="Sem Vítimas") 

In [22]:
# Cria um mapa atribuindo a classificação do acidente ao ID da sua respectiva ocorrência
mapping = df1.set_index("id")["classificacao_acidente"]

# Preenche a coluna com os dados do mapa onde o ID da ocorrência é equivalente
df2["classificacao_acidente"] = df2["id"].map(mapping)

Foram identificadas ainda inconsistências no cálculo do número de pessoas envolvidas na ocorrência, dado registrado no primeiro dataset. Em vários dos registros a soma de pessoas feridas, mortas, ilesas e/ou não registradas não equivale ao valor presente na coluna que determina a quantidade total de pessoas. Para resolver esse problema e garantir a integridade dos dados, todos os cálculos foram refeitos.

In [23]:
# Determina a quantidade de feridos como a soma de feridos leves e graves
df1["feridos"] = df1["feridos_leves"] + df1["feridos_graves"]

# Determina o total de pessoas como a soma de óbitos, feridos, ilesos e não informados
df1["pessoas"] = df1["mortos"] + df1["ilesos"] + df1["ignorados"] + df1["feridos"]

Por fim, as seguintes colunas foram renomeadas:
- O termo "ignorados" foi alterado para "não informados", se referindo aos indivíduos cujo estado físico após o acidente é desconhecido;
- O nome da coluna "condicao_metereologica" foi devidamente corrigido para "condicao_meteorologica";
- A coluna "uso_solo" foi renomeada para "local_acidente", facilitando sua compreensão.

In [24]:
# Altera os nomes das colunas em questão
df1.rename(columns={"ignorados" : "nao_informados", "uso_solo" : "local_acidente", "condicao_metereologica" : "condicao_meteorologica"}, inplace = True)
df2.rename(columns={"uso_solo" : "local_acidente", "condicao_metereologica" : "condicao_meteorologica"}, inplace = True)

### Padronizar e Preparar Dados para Análise

A maneira com que os campos categóricos são representados varia com alguns constituindo strings com todos os caracteres em minúsculo, outros com todos os caracteres maiúsculos, e a maior parte com strings onde apenas o primeiro caractere é maiúsculo. Optou-se por padronizar a forma com que tais campos são representados, mantendo todas as strings com caracteres minúsculos.

In [25]:
# Define as colunas que terão seus campos padronizados
columns_to_lower_df1 = ["municipio", "causa_acidente", "tipo_acidente", "classificacao_acidente", "fase_dia", "sentido_via", "condicao_meteorologica", "tipo_pista", "tracado_via", "local_acidente"]
columns_to_lower_df2 = ["municipio", "causa_acidente", "tipo_acidente", "classificacao_acidente", "fase_dia", "sentido_via", "condicao_meteorologica", "tipo_pista", "tracado_via", "local_acidente", "tipo_veiculo", "tipo_envolvido", "estado_fisico", "sexo"]

# Torna minúsculo todos os caracteres das strings nas colunas especificadas 
df1[columns_to_lower_df1] = df1[columns_to_lower_df1].map(lambda x : x.lower())
df2[columns_to_lower_df2] = df2[columns_to_lower_df2].map(lambda x : x.lower())

A coluna "horario" de ambos os datasets foi formatada, convertendo o conteúdo que anteriormente era uma string para o objeto time, proveniente da biblioteca pandas.

In [26]:
# Formata a coluna "horario", convertendo string em time
df1["horario"] = pd.to_datetime(df1["horario"], format = "%H:%M:%S").dt.time
df2["horario"] = pd.to_datetime(df2["horario"], format = "%H:%M:%S").dt.time

Foram selecionadas as colunas cujos conteúdos não serão úteis no processo de análise, e as mesmas serão removidas dos datasets.

In [27]:
# Remove as colunas em questão
df1.drop(columns = ["km", "regional", "delegacia", "uop"], inplace = True)
df2.drop(columns = ["pesid", "km", "id_veiculo", "marca", "sexo", "ilesos", "feridos_leves", "feridos_graves", "mortos", "regional", "delegacia", "uop"], inplace = True)

A coluna "local_acidente" usa os termos "Sim" e "Não" para representar, respectivamente, rodovias localizadas em zonas urbanas ou rurais. A utilização desses termos é contra-intuitiva e, portanto, eles serão devidamente alterados para seus equivalentes.

In [28]:
# Altera os campos da coluna "local_acidente" seguindo as regras de tradução definidas a seguir
local = {"sim" : "urbano", "não" : "rural"}

df1["local_acidente"] = df1["local_acidente"].map(local)
df2["local_acidente"] = df2["local_acidente"].map(local)

In [29]:
df1

,id,data_inversa,dia_semana,horario,uf,br,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_meteorologica,tipo_pista,tracado_via,local_acidente,pessoas,mortos,feridos_leves,feridos_graves,ilesos,nao_informados,feridos,veiculos,latitude,longitude
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,guarulhos,reação tardia ou ineficiente do condutor,tombamento,com vítimas feridas,pleno dia,decrescente,céu claro,múltipla,reta;declive,urbano,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317"
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,penaforte,pista esburacada,colisão frontal,com vítimas fatais,pleno dia,crescente,céu claro,simples,reta,rural,7,1,1,0,1,4,1,6,"-7,812288","-39,08333306"
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,cornelio procopio,reação tardia ou ineficiente do condutor,colisão traseira,com vítimas feridas,pleno dia,crescente,sol,dupla,reta;aclive,urbano,5,0,3,0,2,0,3,2,"-23,182565","-50,637228"
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,campina grande do sul,reação tardia ou ineficiente do condutor,saída de leito carroçável,com vítimas feridas,pleno dia,crescente,céu claro,dupla,reta,rural,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028"
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,francisco sa,velocidade incompatível,colisão frontal,com vítimas feridas,pleno dia,decrescente,chuva,simples,curva;declive,rural,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72524,757462,2025-12-20,sábado,13:10:00,MG,381,jaguaracu,acessar a via sem observar a presença dos outr...,colisão transversal,com vítimas feridas,pleno dia,decrescente,céu claro,simples,interseção de vias,rural,3,0,2,0,1,0,2,2,"-19,607","-42,7611"
72525,757492,2025-09-27,sábado,13:50:00,SC,282,herval doeste,conversão proibida,colisão transversal,com vítimas feridas,pleno dia,decrescente,céu claro,simples,reta;declive,rural,4,0,1,1,0,2,2,3,"-27,2096","-51,50077"
72526,757593,2025-12-14,domingo,13:50:00,PE,104,caruaru,ausência de reação do condutor,colisão transversal,com vítimas feridas,pleno dia,crescente,céu claro,dupla,reta,urbano,3,0,1,0,2,0,1,2,"-8,25368","-35,97546"
72527,758175,2025-12-15,segunda-feira,15:50:00,SC,101,camboriu,condutor deixou de manter distância do veículo...,colisão traseira,sem vítimas,pleno dia,crescente,sol,dupla,reta,urbano,2,0,0,0,2,0,0,2,"-26,99305955","-48,6609548"


In [30]:
df2

,id,data_inversa,dia_semana,horario,uf,br,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_meteorologica,tipo_pista,tracado_via,local_acidente,tipo_veiculo,ano_fabricacao_veiculo,tipo_envolvido,estado_fisico,idade,latitude,longitude
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,guarulhos,reação tardia ou ineficiente do condutor,tombamento,com vítimas feridas,pleno dia,decrescente,céu claro,múltipla,reta;declive,urbano,caminhão,0,condutor,não informado,0,"-23,48586772","-46,54075317"
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,penaforte,pista esburacada,colisão frontal,com vítimas fatais,pleno dia,crescente,céu claro,simples,reta,rural,caminhão-trator,2021,condutor,ileso,31,"-7,812288","-39,08333306"
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,cornelio procopio,reação tardia ou ineficiente do condutor,colisão traseira,com vítimas feridas,pleno dia,crescente,sol,dupla,reta;aclive,urbano,caminhão,1999,condutor,ileso,60,"-23,182565","-50,637228"
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,campina grande do sul,reação tardia ou ineficiente do condutor,saída de leito carroçável,com vítimas feridas,pleno dia,crescente,céu claro,dupla,reta,rural,caminhão,2013,condutor,ileso,40,"-25,36517687","-49,04223028"
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,francisco sa,velocidade incompatível,colisão frontal,com vítimas feridas,pleno dia,decrescente,chuva,simples,curva;declive,rural,caminhão-trator,2022,condutor,ileso,55,"-16,46801304","-43,43121303"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194624,757593,2025-12-14,domingo,13:50:00,PE,104,caruaru,ausência de reação do condutor,colisão transversal,com vítimas feridas,pleno dia,crescente,céu claro,dupla,reta,urbano,automóvel,2009,condutor,ileso,41,"-8,25368","-35,97546"
194625,758175,2025-12-15,segunda-feira,15:50:00,SC,101,camboriu,condutor deixou de manter distância do veículo...,colisão traseira,sem vítimas,pleno dia,crescente,sol,dupla,reta,urbano,automóvel,2010,condutor,ileso,64,"-26,99305955","-48,6609548"
194626,758175,2025-12-15,segunda-feira,15:50:00,SC,101,camboriu,condutor deixou de manter distância do veículo...,colisão traseira,sem vítimas,pleno dia,crescente,sol,dupla,reta,urbano,caminhonete,2023,condutor,ileso,31,"-26,99305955","-48,6609548"
194627,758186,2025-11-02,domingo,23:00:00,MG,381,belo oriente,acessar a via sem observar a presença dos outr...,colisão traseira,com vítimas feridas,plena noite,decrescente,nublado,simples,reta;interseção de vias,rural,automóvel,2011,condutor,lesões graves,47,"-19,298811","-42,371921"
